# Interseção SICOR Cerrado × Assentamentos por ID_GLEBA2

Este notebook cruza o arquivo `sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv` com o `Assentamento Brasil.shp`.

## Regra principal

- cada `ID_GLEBA2` distinto é submetido à interseção espacial somente uma vez;
- quando o mesmo `ID_GLEBA2` aparece em várias linhas, o resultado espacial calculado é reaproveitado;
- antes do processamento, o notebook valida se um mesmo `ID_GLEBA2` está sempre associado à mesma geometria;
- a deduplicação técnica da interseção usa a chave `ID_GLEBA2 + ASSENT_ID_INTERNO`, e não `REF_BACEN`.

## Saídas

1. **Resultado espacial único:** uma linha por `ID_GLEBA2 × feição de assentamento`.
2. **Resultado detalhado expandido:** associa a resposta espacial a todas as linhas originais que possuem aquele `ID_GLEBA2`.
3. **Resumo por ID_GLEBA2:** contém todos os IDs, inclusive os que não intersectaram assentamento rural.

Assim, uma geometria repetida não é testada novamente, mas nenhum vínculo com `REF_BACEN`, ordem ou ano é perdido.


In [1]:
# ============================================================
# 1. INSTALAR, IMPORTAR E MONTAR O GOOGLE DRIVE
# ============================================================

%pip install -q --upgrade geopandas pyogrio shapely

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import force_2d, intersection as shapely_intersection, make_valid, set_precision
from shapely.errors import GEOSException

warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 180)

print('Ambiente preparado.')


Mounted at /content/drive
Ambiente preparado.


In [2]:
# ============================================================
# 2. CAMINHOS E PARÂMETROS
# ============================================================

ARQUIVO_SICOR = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/"
    "sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv"
)

ARQUIVO_ASSENTAMENTO = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Base/Assentamento Brasil.shp"
)

# Pasta nova para não sobrescrever resultados anteriores.
PASTA_SAIDA = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/intersecoes/assentamentos/"
    "resultados_por_id_gleba2"
)

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_INTERSECAO_POR_ID = (
    PASTA_SAIDA / "intersecao_assentamentos_unica_por_id_gleba2.csv"
)

ARQUIVO_INTERSECAO_EXPANDIDA = (
    PASTA_SAIDA / "intersecao_sicor_assentamentos_expandida_por_id_gleba2.csv"
)

ARQUIVO_RESUMO_POR_ID = (
    PASTA_SAIDA / "resumo_assentamentos_por_id_gleba2.csv"
)

CRS_SICOR = "EPSG:4326"
CRS_AREA = "EPSG:5880"
TAMANHO_BLOCO = 20_000
SEPARADOR_SAIDA = ";"

for caminho in [ARQUIVO_SICOR, ARQUIVO_ASSENTAMENTO]:
    print(f"{'OK' if caminho.exists() else 'NÃO ENCONTRADO'} | {caminho}")

if not ARQUIVO_SICOR.exists():
    raise FileNotFoundError(f"CSV do SICOR não encontrado: {ARQUIVO_SICOR}")

if not ARQUIVO_ASSENTAMENTO.exists():
    raise FileNotFoundError(f"Shapefile de assentamentos não encontrado: {ARQUIVO_ASSENTAMENTO}")

for extensao in [".dbf", ".shx", ".prj"]:
    componente = ARQUIVO_ASSENTAMENTO.with_suffix(extensao)
    print(f"{'OK' if componente.exists() else 'ATENÇÃO'} | {componente.name}")

print("\nPasta de saída:")
print(PASTA_SAIDA)


OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/Assentamento Brasil.shp
OK | Assentamento Brasil.dbf
OK | Assentamento Brasil.shx
OK | Assentamento Brasil.prj

Pasta de saída:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/assentamentos/resultados_por_id_gleba2


In [3]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

CODIFICACOES = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']
SEPARADORES = [';', ',', '\t', '|']


def remover_acentos(texto):
    return ''.join(
        caractere
        for caractere in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(caractere)
    )


def normalizar_nome_coluna(nome):
    nome = remover_acentos(nome).upper().strip().replace('#', '')
    nome = re.sub(r'[^A-Z0-9]+', '_', nome)
    return re.sub(r'_+', '_', nome).strip('_')


def normalizar_colunas(df):
    df = df.copy()
    df.columns = [normalizar_nome_coluna(coluna) for coluna in df.columns]

    if df.columns.duplicated().any():
        duplicadas = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f'A normalização produziu colunas duplicadas: {duplicadas}')

    return df


def detectar_csv(caminho):
    ultimo_erro = None

    for encoding in CODIFICACOES:
        for separador in SEPARADORES:
            try:
                teste = pd.read_csv(
                    caminho,
                    sep=separador,
                    encoding=encoding,
                    nrows=5,
                    dtype=str,
                    low_memory=False,
                )

                if teste.shape[1] <= 1:
                    continue

                teste = normalizar_colunas(teste)
                return encoding, separador, list(teste.columns)

            except Exception as erro:
                ultimo_erro = erro

    raise RuntimeError(
        'Não foi possível detectar a estrutura do CSV. '
        f'Último erro: {ultimo_erro}'
    )


def localizar_coluna(colunas, candidatos, obrigatoria=True):
    for candidato in [normalizar_nome_coluna(c) for c in candidatos]:
        if candidato in colunas:
            return candidato

    if obrigatoria:
        raise KeyError(
            'Nenhuma das colunas esperadas foi encontrada: '
            f'{[normalizar_nome_coluna(c) for c in candidatos]}'
        )

    return None


def padronizar_codigo(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.lstrip('\ufeff')
        .str.replace(r'\.0$', '', regex=True)
        .replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, '<NA>': pd.NA})
    )


def corrigir_geometrias(gdf):
    quantidade_entrada = len(gdf)

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    mascara_invalida = ~gdf.geometry.is_valid
    quantidade_invalidas = int(mascara_invalida.sum())

    if quantidade_invalidas:
        try:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].make_valid()
            )
        except Exception:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].buffer(0)
            )

    gdf = gdf[
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
        & gdf.geometry.is_valid
    ].copy()

    quantidade_descartadas = quantidade_entrada - len(gdf)
    return gdf, quantidade_invalidas, quantidade_descartadas


def serie_wkt_para_geometria(serie):
    return gpd.GeoSeries.from_wkt(
        serie.astype('string'),
        crs=CRS_SICOR,
        on_invalid='ignore',
    )


def preparar_geometria_topologica(geometria, precisao=None):
    if geometria is None:
        return None

    try:
        if geometria.is_empty:
            return None
        geometria = force_2d(geometria)

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        if precisao is not None:
            geometria = set_precision(
                geometria,
                grid_size=precisao,
                mode='valid_output',
            )

        if geometria is None or geometria.is_empty:
            return None

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        return geometria

    except Exception:
        try:
            geometria = geometria.buffer(0)
            return None if geometria.is_empty else geometria
        except Exception:
            return None


def calcular_intersecoes_seguras(geometrias_esquerda, geometrias_direita, crs):
    resultados = []
    quantidade_recuperadas = 0
    quantidade_falhas = 0

    for geometria_esquerda, geometria_direita in zip(
        geometrias_esquerda.to_numpy(),
        geometrias_direita.to_numpy(),
    ):
        esquerda = preparar_geometria_topologica(geometria_esquerda, 0.01)
        direita = preparar_geometria_topologica(geometria_direita, 0.01)

        if esquerda is None or direita is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        intersecao = None
        recuperada = False

        try:
            intersecao = shapely_intersection(esquerda, direita, grid_size=0.01)
        except (GEOSException, ValueError, TypeError):
            recuperada = True

            try:
                esquerda_10 = preparar_geometria_topologica(esquerda, 0.10)
                direita_10 = preparar_geometria_topologica(direita, 0.10)

                if esquerda_10 is not None and direita_10 is not None:
                    intersecao = shapely_intersection(
                        esquerda_10,
                        direita_10,
                        grid_size=0.10,
                    )
            except (GEOSException, ValueError, TypeError):
                intersecao = None

            if intersecao is None:
                try:
                    intersecao = esquerda.buffer(0).intersection(direita.buffer(0))
                except Exception:
                    intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        try:
            if not intersecao.is_empty and not intersecao.is_valid:
                intersecao = make_valid(intersecao)
        except Exception:
            try:
                intersecao = intersecao.buffer(0)
            except Exception:
                intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
        else:
            resultados.append(intersecao)
            if recuperada:
                quantidade_recuperadas += 1

    return (
        gpd.GeoSeries(resultados, index=pd.RangeIndex(len(resultados)), crs=crs),
        quantidade_recuperadas,
        quantidade_falhas,
    )


In [4]:
# ============================================================
# 4. LER E PREPARAR OS ASSENTAMENTOS
# ============================================================

assentamentos = gpd.read_file(ARQUIVO_ASSENTAMENTO, engine='pyogrio')
crs_original_assent = assentamentos.crs
coluna_geometria_original = assentamentos.geometry.name

if crs_original_assent is None:
    raise ValueError(
        'O Assentamento Brasil.shp não possui CRS. Verifique se o arquivo .prj '
        'está na mesma pasta do shapefile.'
    )

assentamentos = normalizar_colunas(assentamentos)
coluna_geometria_normalizada = normalizar_nome_coluna(coluna_geometria_original)

colunas_geometricas = [
    coluna
    for coluna in assentamentos.columns
    if isinstance(assentamentos[coluna].dtype, gpd.array.GeometryDtype)
]

if coluna_geometria_normalizada not in assentamentos.columns:
    if not colunas_geometricas:
        raise KeyError('Nenhuma coluna geométrica foi localizada no shapefile de assentamentos.')
    coluna_geometria_normalizada = colunas_geometricas[0]

assentamentos = gpd.GeoDataFrame(
    assentamentos,
    geometry=coluna_geometria_normalizada,
    crs=crs_original_assent,
)

if assentamentos.geometry.name != 'geometry':
    assentamentos = assentamentos.rename_geometry('geometry')

quantidade_assent_original = len(assentamentos)
assentamentos, invalidas_assent, descartadas_assent = corrigir_geometrias(assentamentos)
assentamentos = assentamentos.reset_index(drop=True)
assentamentos['ASSENT_ID_INTERNO'] = np.arange(1, len(assentamentos) + 1)

renomear_assent = {
    coluna: f'ASSENT_{coluna}'
    for coluna in assentamentos.columns
    if coluna not in {'geometry', 'ASSENT_ID_INTERNO'}
}
assentamentos = assentamentos.rename(columns=renomear_assent).set_geometry('geometry')
assentamentos = assentamentos.to_crs(CRS_SICOR).reset_index(drop=True)

assentamentos_area = assentamentos.to_crs(CRS_AREA)
assentamentos['ASSENT_AREA_TOTAL_HA'] = assentamentos_area.geometry.area / 10_000

COLUNAS_ASSENT_SAIDA = [coluna for coluna in assentamentos.columns if coluna != 'geometry']

print('Feições originais de assentamentos:', f'{quantidade_assent_original:,}')
print('Geometrias inválidas corrigidas:', f'{invalidas_assent:,}')
print('Geometrias descartadas:', f'{descartadas_assent:,}')
print('Feições válidas finais:', f'{len(assentamentos):,}')
print('CRS espacial:', assentamentos.crs)
print('CRS para áreas:', CRS_AREA)

display(assentamentos.head())


Feições originais de assentamentos: 8,214
Geometrias inválidas corrigidas: 0
Geometrias descartadas: 1
Feições válidas finais: 8,213
CRS espacial: EPSG:4326
CRS para áreas: EPSG:5880


,ASSENT_CD_SIPRA,ASSENT_UF,ASSENT_NOME_PROJE,ASSENT_MUNICIPIO,ASSENT_AREA_HECTA,ASSENT_CAPACIDADE,ASSENT_NUM_FAMILI,ASSENT_FASE,ASSENT_DATA_DE_CR,ASSENT_FORMA_OBTE,ASSENT_DATA_OBTEN,ASSENT_AREA_CALC,ASSENT_SR,ASSENT_DESCRICAO,geometry,ASSENT_ID_INTERNO,ASSENT_AREA_TOTAL_HA
0,MS0138000,MS,PA ESTRELA JARAGUARI,JARAGUARI,2009.0000,195,194,3,05/12/2007,Compra e Venda,13/04/2007,2691.1797,None,None,"POLYGON ((-54.29403 -20.4358, -54.29406 -20.43587, -54.29428 -20.43804, -54.29439 -20.43977, -54.29392 -20.44118, -54.2937 -20.44202, -54.29429 -20.44316, -54.29424 -20.444, -5...",1,2690.975390
1,MS0224000,MS,PA INDAIA III - FETAGRI,AQUIDAUANA,1635.0456,72,69,3,28/12/2007,Desapropriação,01/12/2006,1618.7269,None,None,"MULTIPOLYGON (((-55.59963 -20.20481, -55.59958 -20.20484, -55.59646 -20.20694, -55.59638 -20.207, -55.59295 -20.20932, -55.592 -20.20996, -55.5893 -20.2118, -55.58813 -20.21259...",2,1615.354463
2,MS0225000,MS,PA INDAIA IV - MST,AQUIDAUANA,1636.1332,51,51,3,28/12/2007,Desapropriação,01/12/2006,1616.1850,None,None,"MULTIPOLYGON (((-55.5331 -20.20058, -55.53252 -20.20075, -55.53147 -20.20103, -55.52882 -20.20175, -55.52692 -20.20227, -55.52602 -20.20251, -55.52559 -20.20262, -55.52536 -20....",3,1607.369185
3,MS0099000,MS,PA NOVA ERA,PONTA PORA,2802.0089,97,96,3,06/11/2000,Desapropriação,25/10/1999,2804.2007,None,None,"POLYGON ((-55.61524 -22.02629, -55.61431 -22.02639, -55.61364 -22.02646, -55.60585 -22.02731, -55.60573 -22.0273, -55.60557 -22.02728, -55.60481 -22.02719, -55.60477 -22.02926,...",4,2827.416393
4,MB0482000,MB,PA NOVA VIDA II,PACAJA,26924.8690,320,311,3,13/09/2006,Arrecadação,13/09/2006,NaN,None,None,"POLYGON ((-50.18158 -3.74015, -50.18377 -3.7372, -50.17816 -3.73493, -50.17501 -3.73408, -50.1723 -3.73341, -50.16876 -3.73252, -50.16457 -3.73146, -50.16121 -3.73067, -50.1561...",5,25808.995232


In [5]:
# ============================================================
# 5. DETECTAR E VALIDAR AS COLUNAS DO SICOR
# ============================================================

encoding_sicor, separador_sicor, colunas_sicor = detectar_csv(ARQUIVO_SICOR)

COLUNA_ID_GLEBA2 = localizar_coluna(
    colunas_sicor,
    ['ID_GLEBA2', 'ID_GLEBA_2'],
)

COLUNA_GEOMETRIA_SICOR = localizar_coluna(
    colunas_sicor,
    ['GT_GEOMETRIA', 'GT_GEOMETRIA_REPRESENTATIVA', 'GEOMETRIA', 'GEOMETRY', 'WKT'],
)

COLUNA_REF_BACEN = localizar_coluna(
    colunas_sicor,
    ['REF_BACEN', '#REF_BACEN', 'CD_REF_BACEN'],
    obrigatoria=False,
)

print('Codificação:', encoding_sicor)
print('Separador:', repr(separador_sicor))
print('Coluna-chave:', COLUNA_ID_GLEBA2)
print('Coluna de geometria:', COLUNA_GEOMETRIA_SICOR)
print('Coluna REF_BACEN:', COLUNA_REF_BACEN)
print('Quantidade de colunas:', len(colunas_sicor))

if COLUNA_ID_GLEBA2 == COLUNA_GEOMETRIA_SICOR:
    raise RuntimeError('A coluna do ID e a coluna geométrica não podem ser a mesma.')


Codificação: utf-8-sig
Separador: ';'
Coluna-chave: ID_GLEBA2
Coluna de geometria: GT_GEOMETRIA
Coluna REF_BACEN: REF_BACEN
Quantidade de colunas: 27


In [6]:
# ============================================================
# 6. INTERSECTAR UMA ÚNICA VEZ CADA ID_GLEBA2
# ============================================================

# Guarda somente uma assinatura numérica da geometria de cada ID.
# Isso permite reconhecer IDs já processados sem manter todos os WKT na memória.
assinatura_geometria_por_id = {}
resultados_intersecao = []

total_linhas_lidas = 0
total_linhas_com_id_e_geometria = 0
total_ids_novos = 0
total_repeticoes_puladas = 0
total_ids_geometria_valida = 0
total_geometrias_invalidas_corrigidas = 0
total_geometrias_descartadas = 0
total_candidatos_espaciais = 0
total_intersecoes_validas = 0
total_intersecoes_recuperadas = 0
total_falhas_topologicas = 0
total_contatos_sem_area = 0

leitor = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor, start=1):
    bloco = normalizar_colunas(bloco)
    total_linhas_lidas += len(bloco)

    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    bloco[COLUNA_GEOMETRIA_SICOR] = bloco[COLUNA_GEOMETRIA_SICOR].astype('string').str.strip()

    bloco = bloco.dropna(
        subset=[COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
    ).copy()

    total_linhas_com_id_e_geometria += len(bloco)

    if bloco.empty:
        print(f'Bloco {numero_bloco:03d}: sem ID_GLEBA2 e geometria válidos')
        continue

    bloco['_HASH_GEOMETRIA'] = (
        pd.util.hash_pandas_object(
            bloco[COLUNA_GEOMETRIA_SICOR],
            index=False,
        )
        .astype('uint64')
    )

    # Validação dentro do próprio bloco.
    geometrias_por_id_no_bloco = bloco.groupby(
        COLUNA_ID_GLEBA2,
        dropna=False,
    )['_HASH_GEOMETRIA'].nunique()

    ids_inconsistentes = geometrias_por_id_no_bloco[
        geometrias_por_id_no_bloco > 1
    ].index.tolist()

    if ids_inconsistentes:
        raise ValueError(
            'Um mesmo ID_GLEBA2 foi associado a geometrias diferentes. '
            f'Primeiros IDs inconsistentes: {ids_inconsistentes[:10]}'
        )

    representantes = bloco.drop_duplicates(
        subset=[COLUNA_ID_GLEBA2],
        keep='first',
    ).reset_index(drop=True)

    mascara_id_novo = []

    # Validação também contra IDs que apareceram em blocos anteriores.
    for id_atual, hash_atual in representantes[
        [COLUNA_ID_GLEBA2, '_HASH_GEOMETRIA']
    ].itertuples(index=False, name=None):
        hash_atual = int(hash_atual)

        if id_atual in assinatura_geometria_por_id:
            if assinatura_geometria_por_id[id_atual] != hash_atual:
                raise ValueError(
                    f'O ID_GLEBA2 {id_atual} apareceu com geometrias diferentes '
                    'em blocos distintos do CSV.'
                )
            mascara_id_novo.append(False)
        else:
            assinatura_geometria_por_id[id_atual] = hash_atual
            mascara_id_novo.append(True)

    representantes_novos = representantes.loc[
        mascara_id_novo
    ].copy().reset_index(drop=True)

    total_ids_novos += len(representantes_novos)
    total_repeticoes_puladas += len(bloco) - len(representantes_novos)

    if representantes_novos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            'todos os IDs já haviam sido processados'
        )
        del bloco, representantes, representantes_novos
        gc.collect()
        continue

    geometria = serie_wkt_para_geometria(
        representantes_novos[COLUNA_GEOMETRIA_SICOR]
    )

    gdf_ids = gpd.GeoDataFrame(
        representantes_novos[
            [COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
        ].copy(),
        geometry=geometria,
        crs=CRS_SICOR,
    )

    gdf_ids, invalidas_corrigidas, descartadas = corrigir_geometrias(gdf_ids)
    gdf_ids = gdf_ids.reset_index(drop=True)

    total_geometrias_invalidas_corrigidas += invalidas_corrigidas
    total_geometrias_descartadas += descartadas
    total_ids_geometria_valida += len(gdf_ids)

    if gdf_ids.empty:
        print(
            f'Bloco {numero_bloco:03d}: IDs novos={len(representantes_novos):,} | '
            'nenhuma geometria válida'
        )
        del bloco, representantes, representantes_novos, gdf_ids
        gc.collect()
        continue

    candidatos = gpd.sjoin(
        gdf_ids,
        assentamentos,
        how='inner',
        predicate='intersects',
    ).reset_index(drop=True)

    total_candidatos_espaciais += len(candidatos)

    if candidatos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            f'IDs novos={len(representantes_novos):,} | interseções=0 | '
            f'repetições puladas={len(bloco) - len(representantes_novos):,}'
        )
        del bloco, representantes, representantes_novos, gdf_ids, candidatos
        gc.collect()
        continue

    candidatos['GEOMETRIA_ASSENTAMENTO'] = (
        candidatos['index_right'].map(assentamentos.geometry)
    )

    candidatos = candidatos.dropna(
        subset=['GEOMETRIA_ASSENTAMENTO']
    ).reset_index(drop=True)

    geometria_gleba_wgs84 = gpd.GeoSeries(
        candidatos.geometry.to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_assent_wgs84 = gpd.GeoSeries(
        candidatos['GEOMETRIA_ASSENTAMENTO'].to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_gleba_area = geometria_gleba_wgs84.to_crs(CRS_AREA)
    geometria_assent_area = geometria_assent_wgs84.to_crs(CRS_AREA)

    geometria_gleba_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_gleba_area],
        index=pd.RangeIndex(len(geometria_gleba_area)),
        crs=CRS_AREA,
    )

    geometria_assent_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_assent_area],
        index=pd.RangeIndex(len(geometria_assent_area)),
        crs=CRS_AREA,
    )

    (
        geometria_intersecao_area,
        recuperadas_bloco,
        falhas_bloco,
    ) = calcular_intersecoes_seguras(
        geometria_gleba_area,
        geometria_assent_area,
        CRS_AREA,
    )

    total_intersecoes_recuperadas += recuperadas_bloco
    total_falhas_topologicas += falhas_bloco

    candidatos['AREA_GLEBA_HA_CALCULADA'] = (
        geometria_gleba_area.area.to_numpy() / 10_000
    )
    candidatos['AREA_INTERSECAO_HA'] = (
        geometria_intersecao_area.area.to_numpy() / 10_000
    )
    candidatos['PERCENTUAL_GLEBA_INTERSECTADA'] = np.where(
        candidatos['AREA_GLEBA_HA_CALCULADA'] > 0,
        candidatos['AREA_INTERSECAO_HA']
        / candidatos['AREA_GLEBA_HA_CALCULADA']
        * 100,
        np.nan,
    )

    mascara_area_positiva = (
        candidatos['AREA_INTERSECAO_HA'].fillna(0).gt(0).to_numpy()
    )
    total_contatos_sem_area += int((~mascara_area_positiva).sum())

    if not mascara_area_positiva.any():
        print(
            f'Bloco {numero_bloco:03d}: candidatos={len(candidatos):,} | '
            'nenhuma interseção com área positiva'
        )
        del (
            bloco, representantes, representantes_novos, gdf_ids, candidatos,
            geometria_gleba_wgs84, geometria_assent_wgs84,
            geometria_gleba_area, geometria_assent_area,
            geometria_intersecao_area, mascara_area_positiva,
        )
        gc.collect()
        continue

    candidatos = candidatos.loc[mascara_area_positiva].reset_index(drop=True)

    geometria_intersecao_area_valida = gpd.GeoSeries(
        geometria_intersecao_area.to_numpy()[mascara_area_positiva],
        index=pd.RangeIndex(int(mascara_area_positiva.sum())),
        crs=CRS_AREA,
    )
    geometria_intersecao_wgs84 = geometria_intersecao_area_valida.to_crs(CRS_SICOR)

    candidatos['GEOMETRIA_ASSENTAMENTO_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in candidatos['GEOMETRIA_ASSENTAMENTO']
    ]
    candidatos['GEOMETRIA_INTERSECAO_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in geometria_intersecao_wgs84
    ]
    candidatos['TIPO_IMPEDIMENTO'] = 'Assentamento Rural'

    colunas_resultado = [
        COLUNA_ID_GLEBA2,
        *COLUNAS_ASSENT_SAIDA,
        'AREA_GLEBA_HA_CALCULADA',
        'AREA_INTERSECAO_HA',
        'PERCENTUAL_GLEBA_INTERSECTADA',
        'GEOMETRIA_ASSENTAMENTO_WKT',
        'GEOMETRIA_INTERSECAO_WKT',
        'TIPO_IMPEDIMENTO',
    ]

    resultado_bloco = pd.DataFrame(candidatos[colunas_resultado])
    resultados_intersecao.append(resultado_bloco)
    total_intersecoes_validas += len(resultado_bloco)

    print(
        f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
        f'IDs novos={len(representantes_novos):,} | '
        f'repetições puladas={len(bloco) - len(representantes_novos):,} | '
        f'candidatos={len(mascara_area_positiva):,} | '
        f'interseções com área={len(resultado_bloco):,}'
    )

    del (
        bloco, representantes, representantes_novos, gdf_ids, candidatos,
        geometria_gleba_wgs84, geometria_assent_wgs84,
        geometria_gleba_area, geometria_assent_area,
        geometria_intersecao_area, geometria_intersecao_area_valida,
        geometria_intersecao_wgs84, mascara_area_positiva, resultado_bloco,
    )
    gc.collect()

print('\nPROCESSAMENTO ESPACIAL CONCLUÍDO')
print('=' * 90)
print('Linhas lidas:', f'{total_linhas_lidas:,}')
print('Linhas com ID e geometria:', f'{total_linhas_com_id_e_geometria:,}')
print('ID_GLEBA2 distintos:', f'{len(assinatura_geometria_por_id):,}')
print('IDs submetidos ao teste:', f'{total_ids_novos:,}')
print('Repetições que não foram testadas novamente:', f'{total_repeticoes_puladas:,}')
print('IDs com geometria válida:', f'{total_ids_geometria_valida:,}')
print('Geometrias inválidas corrigidas:', f'{total_geometrias_invalidas_corrigidas:,}')
print('Geometrias descartadas:', f'{total_geometrias_descartadas:,}')
print('Candidatos espaciais:', f'{total_candidatos_espaciais:,}')
print('Interseções recuperadas:', f'{total_intersecoes_recuperadas:,}')
print('Falhas topológicas:', f'{total_falhas_topologicas:,}')
print('Contatos sem área positiva:', f'{total_contatos_sem_area:,}')
print('Interseções válidas ID_GLEBA2 × assentamento:', f'{total_intersecoes_validas:,}')


Bloco 001: lidas=20,000 | IDs novos=18,663 | repetições puladas=1,337 | candidatos=1,326 | interseções com área=1,326
Bloco 002: lidas=20,000 | IDs novos=17,690 | repetições puladas=2,310 | candidatos=974 | interseções com área=974
Bloco 003: lidas=20,000 | IDs novos=17,256 | repetições puladas=2,744 | candidatos=1,416 | interseções com área=1,416
Bloco 004: lidas=20,000 | IDs novos=17,061 | repetições puladas=2,939 | candidatos=977 | interseções com área=977
Bloco 005: lidas=20,000 | IDs novos=16,706 | repetições puladas=3,294 | candidatos=964 | interseções com área=964
Bloco 006: lidas=20,000 | IDs novos=14,364 | repetições puladas=5,636 | candidatos=966 | interseções com área=963
Bloco 007: lidas=20,000 | IDs novos=15,135 | repetições puladas=4,865 | candidatos=562 | interseções com área=561
Bloco 008: lidas=20,000 | IDs novos=15,230 | repetições puladas=4,770 | candidatos=826 | interseções com área=825
Bloco 009: lidas=20,000 | IDs novos=14,954 | repetições puladas=5,046 | candidat

In [7]:
# ============================================================
# 7. CONSOLIDAR E SALVAR O RESULTADO ÚNICO POR ID_GLEBA2
# ============================================================

if resultados_intersecao:
    intersecoes_por_id = pd.concat(
        resultados_intersecao,
        ignore_index=True,
        sort=False,
    )
else:
    intersecoes_por_id = pd.DataFrame(
        columns=[COLUNA_ID_GLEBA2, 'ASSENT_ID_INTERNO']
    )

del resultados_intersecao
gc.collect()

quantidade_antes = len(intersecoes_por_id)

if not intersecoes_por_id.empty:
    intersecoes_por_id = (
        intersecoes_por_id
        .sort_values('AREA_INTERSECAO_HA', ascending=False)
        .drop_duplicates(
            subset=[COLUNA_ID_GLEBA2, 'ASSENT_ID_INTERNO'],
            keep='first',
        )
        .reset_index(drop=True)
    )

print(
    'Duplicações técnicas ID_GLEBA2 × assentamento removidas:',
    f'{quantidade_antes - len(intersecoes_por_id):,}',
)

intersecoes_por_id.to_csv(
    ARQUIVO_INTERSECAO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

# O resumo inclui todos os IDs, inclusive os sem interseção.
todos_ids = pd.DataFrame({
    COLUNA_ID_GLEBA2: list(assinatura_geometria_por_id.keys())
})

if intersecoes_por_id.empty:
    resumo_por_id = todos_ids.copy()
    resumo_por_id['TEM_INTERSECAO_ASSENTAMENTO'] = False
    resumo_por_id['QUANTIDADE_ASSENTAMENTOS_INTERSECTADOS'] = 0
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = 0.0
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = 0.0
else:
    resumo_positivo = (
        intersecoes_por_id.groupby(COLUNA_ID_GLEBA2, dropna=False)
        .agg(
            QUANTIDADE_ASSENTAMENTOS_INTERSECTADOS=(
                'ASSENT_ID_INTERNO', 'nunique'
            ),
            AREA_INTERSECAO_TOTAL_HA=('AREA_INTERSECAO_HA', 'sum'),
            PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA=(
                'PERCENTUAL_GLEBA_INTERSECTADA', 'max'
            ),
        )
        .reset_index()
    )

    resumo_por_id = todos_ids.merge(
        resumo_positivo,
        on=COLUNA_ID_GLEBA2,
        how='left',
        validate='one_to_one',
    )

    resumo_por_id['TEM_INTERSECAO_ASSENTAMENTO'] = (
        resumo_por_id['QUANTIDADE_ASSENTAMENTOS_INTERSECTADOS']
        .notna()
    )
    resumo_por_id['QUANTIDADE_ASSENTAMENTOS_INTERSECTADOS'] = (
        resumo_por_id['QUANTIDADE_ASSENTAMENTOS_INTERSECTADOS']
        .fillna(0)
        .astype('int64')
    )
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = (
        resumo_por_id['AREA_INTERSECAO_TOTAL_HA'].fillna(0.0)
    )
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = (
        resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'].fillna(0.0)
    )

    resumo_por_id = resumo_por_id[
        [
            COLUNA_ID_GLEBA2,
            'TEM_INTERSECAO_ASSENTAMENTO',
            'QUANTIDADE_ASSENTAMENTOS_INTERSECTADOS',
            'AREA_INTERSECAO_TOTAL_HA',
            'PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA',
        ]
    ]

resumo_por_id.to_csv(
    ARQUIVO_RESUMO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

print('\nResultado único salvo em:')
print(ARQUIVO_INTERSECAO_POR_ID)
print('\nResumo de todos os IDs salvo em:')
print(ARQUIVO_RESUMO_POR_ID)
print('\nIDs com interseção:', f"{resumo_por_id['TEM_INTERSECAO_ASSENTAMENTO'].sum():,}")
print('IDs sem interseção:', f"{(~resumo_por_id['TEM_INTERSECAO_ASSENTAMENTO']).sum():,}")


Duplicações técnicas ID_GLEBA2 × assentamento removidas: 0

Resultado único salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/assentamentos/resultados_por_id_gleba2/intersecao_assentamentos_unica_por_id_gleba2.csv

Resumo de todos os IDs salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/assentamentos/resultados_por_id_gleba2/resumo_assentamentos_por_id_gleba2.csv

IDs com interseção: 24,615
IDs sem interseção: 370,960


In [8]:
# ============================================================
# 8. DEVOLVER A RESPOSTA A TODAS AS LINHAS DO SICOR
# ============================================================

# A interseção não é recalculada aqui. O merge apenas replica a resposta
# de cada ID_GLEBA2 para todos os REF_BACEN, ordens, índices e anos ligados a ele.

colunas_espaciais_para_merge = [
    coluna
    for coluna in intersecoes_por_id.columns
    if coluna != COLUNA_ID_GLEBA2
]

escrever_cabecalho = True
total_linhas_originais_reprocessadas = 0
total_linhas_expandidas = 0

leitor_expandir = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor_expandir, start=1):
    bloco = normalizar_colunas(bloco)
    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    total_linhas_originais_reprocessadas += len(bloco)

    if intersecoes_por_id.empty:
        expandido = pd.DataFrame()
    else:
        expandido = bloco.merge(
            intersecoes_por_id,
            on=COLUNA_ID_GLEBA2,
            how='inner',
            validate='many_to_many',
            suffixes=('', '_INTERSECAO'),
        )

    if expandido.empty:
        print(f'Bloco {numero_bloco:03d}: nenhuma linha com interseção')
        del bloco, expandido
        gc.collect()
        continue

    expandido.to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        mode='w' if escrever_cabecalho else 'a',
        header=escrever_cabecalho,
        index=False,
        sep=SEPARADOR_SAIDA,
        decimal=',',
        encoding='utf-8-sig' if escrever_cabecalho else 'utf-8',
    )

    total_linhas_expandidas += len(expandido)
    escrever_cabecalho = False

    print(
        f'Bloco {numero_bloco:03d}: originais={len(bloco):,} | '
        f'linhas de interseção expandidas={len(expandido):,}'
    )

    del bloco, expandido
    gc.collect()

if escrever_cabecalho:
    # Nenhuma interseção: ainda cria um CSV vazio com cabeçalho conhecido.
    pd.DataFrame(
        columns=[*colunas_sicor, *colunas_espaciais_para_merge]
    ).to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        index=False,
        sep=SEPARADOR_SAIDA,
        encoding='utf-8-sig',
    )

if total_linhas_originais_reprocessadas != total_linhas_lidas:
    raise RuntimeError(
        'A quantidade de linhas relidas para a expansão difere da '
        'quantidade lida na etapa espacial.'
    )

print('\nRESULTADO DETALHADO EXPANDIDO')
print('=' * 90)
print('Linhas originais relidas:', f'{total_linhas_originais_reprocessadas:,}')
print('Linhas finais de interseção:', f'{total_linhas_expandidas:,}')
print('Arquivo salvo em:')
print(ARQUIVO_INTERSECAO_EXPANDIDA)


Bloco 001: originais=20,000 | linhas de interseção expandidas=1,451
Bloco 002: originais=20,000 | linhas de interseção expandidas=1,171
Bloco 003: originais=20,000 | linhas de interseção expandidas=1,791
Bloco 004: originais=20,000 | linhas de interseção expandidas=1,317
Bloco 005: originais=20,000 | linhas de interseção expandidas=1,294
Bloco 006: originais=20,000 | linhas de interseção expandidas=1,601
Bloco 007: originais=20,000 | linhas de interseção expandidas=937
Bloco 008: originais=20,000 | linhas de interseção expandidas=1,354
Bloco 009: originais=20,000 | linhas de interseção expandidas=1,434
Bloco 010: originais=20,000 | linhas de interseção expandidas=1,866
Bloco 011: originais=20,000 | linhas de interseção expandidas=1,246
Bloco 012: originais=20,000 | linhas de interseção expandidas=1,341
Bloco 013: originais=20,000 | linhas de interseção expandidas=1,589
Bloco 014: originais=20,000 | linhas de interseção expandidas=1,676
Bloco 015: originais=20,000 | linhas de interseção

In [9]:
# ============================================================
# 9. CONFERÊNCIA FINAL
# ============================================================

print('ARQUIVOS GERADOS')
print('=' * 100)

for caminho in [
    ARQUIVO_INTERSECAO_POR_ID,
    ARQUIVO_INTERSECAO_EXPANDIDA,
    ARQUIVO_RESUMO_POR_ID,
]:
    print(f"{'OK' if caminho.exists() else 'NÃO CRIADO'} | {caminho}")

print('\nAMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2')
display(intersecoes_por_id.head(5))

print('\nAMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO')
display(resumo_por_id.head(5))

print('\nESTATÍSTICAS')
print('ID_GLEBA2 distintos analisados:', f'{len(resumo_por_id):,}')
print('ID_GLEBA2 com interseção:', f"{resumo_por_id['TEM_INTERSECAO_ASSENTAMENTO'].sum():,}")
print('Pares ID_GLEBA2 × assentamento rural:', f'{len(intersecoes_por_id):,}')

if not intersecoes_por_id.empty:
    print(
        'Área total das interseções únicas (ha):',
        f"{intersecoes_por_id['AREA_INTERSECAO_HA'].sum():,.4f}",
    )


ARQUIVOS GERADOS
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/assentamentos/resultados_por_id_gleba2/intersecao_assentamentos_unica_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/assentamentos/resultados_por_id_gleba2/intersecao_sicor_assentamentos_expandida_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/assentamentos/resultados_por_id_gleba2/resumo_assentamentos_por_id_gleba2.csv

AMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2


,ID_GLEBA2,ASSENT_CD_SIPRA,ASSENT_UF,ASSENT_NOME_PROJE,ASSENT_MUNICIPIO,ASSENT_AREA_HECTA,ASSENT_CAPACIDADE,ASSENT_NUM_FAMILI,ASSENT_FASE,ASSENT_DATA_DE_CR,ASSENT_FORMA_OBTE,ASSENT_DATA_OBTEN,ASSENT_AREA_CALC,ASSENT_SR,ASSENT_DESCRICAO,ASSENT_ID_INTERNO,ASSENT_AREA_TOTAL_HA,AREA_GLEBA_HA_CALCULADA,AREA_INTERSECAO_HA,PERCENTUAL_GLEBA_INTERSECTADA,GEOMETRIA_ASSENTAMENTO_WKT,GEOMETRIA_INTERSECAO_WKT,TIPO_IMPEDIMENTO
0,243415,MA0061000,MA,PA COCEIRA/NOVA ALEGRIA,TUNTUM,15421.0000,406,404,5,23/05/1994,Desapropriação,23/02/1990,15824.1309,None,None,990,16024.594189,941.230493,689.943405,73.302279,"POLYGON ((-44.82962833 -5.690578329999999, -44.81773167 -5.713712779999996, -44.81765806 -5.713860279999996, -44.81079972 -5.728225830000003, -44.80861333 -5.7308736100000015, ...","POLYGON ((-44.82007007562066 -5.716484235084435, -44.826688071990034 -5.723789529057554, -44.8411760283675 -5.712033199248997, -44.8538628405708 -5.703941719714771, -44.8467090...",Assentamento Rural
1,3549,MA0574000,MA,PA LAGOA DA FORTUNA,FORTUNA,21476.6065,648,648,5,05/10/2001,Desapropriação,26/06/2000,30467.1109,None,None,6092,30919.557414,660.210104,660.210104,100.000000,"POLYGON ((-44.222981517244634 -5.707548265606147, -44.222152937307975 -5.708201340882796, -44.22168258605428 -5.708571229977453, -44.198073574913835 -5.721914716309771, -44.197...","POLYGON ((-44.05931503926928 -5.754000031341446, -44.05161403545303 -5.739395963792555, -44.037859006091914 -5.748290988054308, -44.03507102276728 -5.75033100351225, -44.028962...",Assentamento Rural
2,97958,DF0049000,DF,PA PALMITAL,SAO JOAO D'ALIANCA,838.7830,33,33,5,25/11/1998,Desapropriação,19/11/1997,838.6469,None,None,1861,843.893695,1978.237482,648.130743,32.763040,"POLYGON ((-47.38573 -14.78634306, -47.38669083 -14.79344667, -47.38749528 -14.79969444, -47.38367722 -14.80026417, -47.38352389 -14.79814472, -47.38136056 -14.79850278, -47.380...","POLYGON ((-47.37358496968854 -14.801592020982026, -47.379713035927686 -14.801886999916686, -47.38202700073626 -14.806048015758908, -47.38718627021173 -14.81027523499519, -47.38...",Assentamento Rural
3,2086,GO0190000,GO,PA SANTO ANTÔNIO DAS AREIAS,FAINA,1227.5658,24,24,4,22/02/2001,Compra e Venda,13/11/1997,1231.8765,None,None,4899,1234.118201,628.043750,609.523017,97.051044,"POLYGON ((-50.424380329529335 -15.430109594574882, -50.41925412410433 -15.43055387300992, -50.41378878471096 -15.431015127585479, -50.41047526710322 -15.432007313468482, -50.40...","POLYGON ((-50.42916401104738 -15.440238001790584, -50.429024992994776 -15.439927031367066, -50.428889031530176 -15.439861962759151, -50.42872604177109 -15.439672044817913, -50....",Assentamento Rural
4,76026,GO0363000,GO,PA FERNANDO SILVA,PORANGATU,5465.0284,94,83,3,22/11/2007,Compra e Venda,18/09/2007,5464.3935,None,None,7542,5481.135661,608.701347,608.701347,100.000000,"POLYGON ((-49.39571806 -13.59702778, -49.39402861 -13.59722167, -49.39218083 -13.59770639, -49.38863917 -13.59739083, -49.38719278 -13.59764528, -49.3871425 -13.59781583, -49.3...","POLYGON ((-49.37140704573063 -13.653387021743145, -49.38043097398014 -13.673318987707209, -49.40262704620971 -13.662725964459872, -49.38892496873014 -13.643383035837848, -49.37...",Assentamento Rural



AMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO


,ID_GLEBA2,TEM_INTERSECAO_ASSENTAMENTO,QUANTIDADE_ASSENTAMENTOS_INTERSECTADOS,AREA_INTERSECAO_TOTAL_HA,PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA
0,1,False,0,0.0,0.0
1,2,False,0,0.0,0.0
2,3,False,0,0.0,0.0
3,4,False,0,0.0,0.0
4,5,False,0,0.0,0.0



ESTATÍSTICAS
ID_GLEBA2 distintos analisados: 395,575
ID_GLEBA2 com interseção: 24,615
Pares ID_GLEBA2 × assentamento rural: 25,074
Área total das interseções únicas (ha): 459,364.4587
